## What's going on with the LLM? 
Sometimes, things don't work out as expected. And yes, that's to be expected, especially for a library in early stages like Toponymy.

But how can you check what kind of unexpected you're dealing with? Here are some observability tools that you can use to figure out what's going on under the hood and debug issues. 

* **Connectivity Check**: For checking the connectivity status of the underlying LLM
* **Debugging Callbacks**: For observing the payloads and errors of individual "prompt -> call -> response" tasks
* **LiteLLM Callbacks**: Gives access to all of the LiteLLM callback hooks when any LLMNamer backed by LiteLLM is used (useful for assessing cost in terms of tokens and $$ for example)

In [1]:
import json
import numpy as np
from pathlib import Path

## Connectivity Check
Once you've selected your LLM, run a connectivity check. This sends a basic prompt to the LLM and checks that it sends back a response. 

In [2]:
from toponymy.llm_wrappers import OpenAINamer

namer = OpenAINamer()
namer.connectivity_status()

17:40:35 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
17:40:35 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


{'success': True,
 'model': 'openai/gpt-4o-mini',
 'wrapper': 'LiteLLMNamer',
 'response': '{"status": "ok"}',
 'error_type': None,
 'error_message': None,
 'original_exception': None}

## Debug Callback
At the next level, you can inspect the "prompt -> call -> response" through a callback hook. Here's an example event: 
```json
{
  "event": "llm_call_success",
  "model": "openai/gpt-4o-mini",
  "prompt": {
    "system": "You are an expert at classifying newsgroup posts...",
    "user": "Here is the information about the group of newsgroup posts..."
  },
  "prompt_type": "system",
  "raw_response": "{\"topic_name\":\"FBI's Role in Branch Davidian Fire Controversy and Evidence Disputes\",\"topic_specificity\":0.9}",
  "routine": "generate_topic_names",
  "wrapper": "LiteLLMNamer"
}
```

For convenience a small `BasicDebugLogger` utility is included for low-volume local debugging. High-volume use cases should swap in a buffered or queue-backed callback. The `BasicDebugLogger` will continue to append to an existing file, so we clear it here before we run. 

In [3]:
from toponymy.debug_logging import BasicDebugLogger
from toponymy.tools.notebook_data_load import notebook_output_dir

debug_logger_file = notebook_output_dir() / "llm_debug.jsonl"
debug_logger_file.write_text("", encoding="utf-8")  # Clear the file before starting
debug_logger = BasicDebugLogger(debug_logger_file, truncate=True)

namer = OpenAINamer(callback=debug_logger)

We'll run an example below, but first let's also add a LiteLLM callback.

## LiteLLM Callback Hooks

If your chosen namer uses LiteLLM under the hood, you can also use LiteLLM's callback hooks. Note that not every namer is LiteLLM-backed, and this only applies to LiteLLM-backed namers.

You can integrate third-party observability tools like sentry, lunary, or langfuse or implement your own callback. You can check out the LiteLLM docs to see what's possible: 
* 3rd party integrations: https://docs.litellm.ai/docs/observability/callbacks
* Custom callbacks: https://docs.litellm.ai/docs/observability/custom_callback
* Payload spec: https://docs.litellm.ai/docs/proxy/logging_spec


Here's an example using the success callback for logging basic usage data. 

In [4]:
def make_local_success_logger(log_path: str | Path):
    """
    Success logger factory for LiteLLM that logs usage information to a local file.

    Args:
        log_path (str | Path): The path to the local log file.

    Returns:
        Callable: A success logger function for LiteLLM.
    """
    log_path = Path(log_path)

    def local_success_logger(kwargs, completion_response, start_time, end_time):
        usage = getattr(completion_response, "usage", None)
        record = {
            "model": kwargs.get("model"),
            "prompt_tokens": getattr(usage, "prompt_tokens", None) if usage else None,
            "completion_tokens": getattr(usage, "completion_tokens", None) if usage else None,
            "total_tokens": getattr(usage, "total_tokens", None) if usage else None,
            "response_cost": kwargs.get("response_cost"),
        }
        with log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
    return local_success_logger



Since the LiteLLM success callback is a global process state, so you'll need to clear, reset or account for it if doing multiple runs in a session. In our example, we'll add our callback to the existing list of success callbacks and  only attach it around the `.fit` call. Let's get to it. 

## Putting it together
Let's run a quick toy example to see what happens. 

In [5]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()

In [6]:
from sentence_transformers import SentenceTransformer
from toponymy.toponymy import Toponymy, KeyphraseBuilder

topic_model = Toponymy(
    llm_wrapper=namer,
    text_embedding_model=SentenceTransformer("paraphrase-MiniLM-L3-v2"),
    keyphrase_builder=KeyphraseBuilder(ngram_range=(1,6), max_features=15_000, verbose=True),
    object_description="newsgroup posts",
    corpus_description="20-newsgroups dataset",
    exemplar_delimiters=["<EXAMPLE_POST>\n","\n</EXAMPLE_POST>\n\n"],
)

In [7]:
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

Finally, we're at the point where we'll attach the LiteLLM success callback (and play it safe!). And since it will append to any existing file, let's also clear the log file first. 

In [8]:
%%time
import litellm

litellm_logger_file = notebook_output_dir() / "llm_usage.jsonl"
litellm_logger_file.write_text("", encoding="utf-8")  # Clear the file before starting

previous_success_callbacks = list(litellm.success_callback or [])
try: 
    litellm.success_callback = previous_success_callbacks + [make_local_success_logger(litellm_logger_file)]
    topic_model.fit(newsgroups_df["post"].str.strip().values, embeddings, document_map)
finally:
    litellm.success_callback = previous_success_callbacks

Layer 0 found 8 clusters
Building keyphrase matrix ... 
Chunking into 1 chunks of size 20000 for keyphrase identification.
Combining count dictionaries ...
Found 3421 keyphrases.
Chunking into 1 chunks of size 20000 for keyphrase count construction.
Combining count matrix chunks ...


Selecting central exemplars:   0%|          | 0/8 [00:00<?, ?cluster/s]

Batches:   0%|          | 0/107 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/1 [00:00<?, ?layer/s]

Generating informative keyphrases:   0%|          | 0/8 [00:00<?, ?cluster/s]

Generating prompts for layer 0:   0%|          | 0/8 [00:00<?, ?topic/s]

Generating topic names for layer 0:   0%|          | 0/8 [00:00<?, ?topic/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

CPU times: user 3.59 s, sys: 373 ms, total: 3.97 s
Wall time: 12 s


In [9]:
def load_multiline_json_logs(path: str | Path) -> list[dict]:
    """
    Helper function to load a JSONL file. 
    
    Args:
        path (str | Path): Path to the JSONL file.

    Returns:
        list[dict]: List of JSON objects loaded from the file.
    """
    path = Path(path)
    records = []

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping bad JSON on line {i}: {e}")
    return records

First, inspect the debug logs

In [10]:
debug_logs = load_multiline_json_logs(debug_logger_file)

In [11]:
debug_logs[0] if debug_logs else {"message": "No debug log records found"}

{'wrapper': 'LiteLLMNamer',
 'model': 'openai/gpt-4o-mini',
 'event': 'llm_call_success',
 'prompt_type': 'system',
 'routine': 'generate_topic_name',
 'prompt': {'system': '\nYou are an expert at classifying newsgroup posts from 20-newsgroups dataset into topics.\nYour task is to analyze information about a group of newsgroup posts and assign a domain expert level (8 to 15 word) name to this group.\nThe response must be in JSON formatted as {"topic_name":<NAME>, "topic_specificity":<SCORE>}\nwhere NAME is the topic name you generate and SCORE is a float value between 0.0 and 1.0,\nrepresenting how specific and well-defined the topic name is given the input information.\nA score of 1.0 means a perfectly descriptive and specific name, while 0.0 would be a completely generic or unrelated name.\n\n\nEnsure your entire response is only the JSON object, with no other text before or after it.',
  'user': '\nHere is the information about the group of newsgroup posts:\n\n- Keywords for this gr

Now, take a look at the usage logs

In [12]:
litellm_logs = load_multiline_json_logs(litellm_logger_file)

In [13]:
litellm_logs[0] if litellm_logs else {"message": "No LiteLLM usage records found"}

{'model': 'gpt-4o-mini',
 'prompt_tokens': 9261,
 'completion_tokens': 22,
 'total_tokens': 9283,
 'response_cost': 0.00071115}

To get run-level usage sum over the records

In [14]:
def summarize_litellm_logs(records: list[dict]) -> dict:
    """
    Total the number of calls, prompt tokens, completion tokens, total tokens, and total cost from a list of LiteLLM log records.

    Args:
        records (list[dict]): A list of LiteLLM log records.

    Returns:
        dict: A dictionary containing the total counts and cost.
    """
    return {
        "total_calls": len(records),
        "total_prompt_tokens": sum(r.get("prompt_tokens") or 0 for r in records),
        "total_completion_tokens": sum(r.get("completion_tokens") or 0 for r in records),
        "total_tokens": sum(r.get("total_tokens") or 0 for r in records),
        "total_cost": sum(r.get("response_cost") or 0 for r in records),
    }

In [15]:
summarize_litellm_logs(litellm_logs)

{'total_calls': 8,
 'total_prompt_tokens': 27184,
 'total_completion_tokens': 187,
 'total_tokens': 27371,
 'total_cost': 0.0022026}